# `df.style`

possible API:

We'd like to conditionally format a dataframe. We'd like to be able to format cells based on

- The cells value
- The cells value relative to the column

(these may be incompatible) not sure. The actual styling will be CSS selectors/properties; that's what your function should return.

The API should be somthing like

```python
df.style(rows={'A': func1, 'B': func2}, columns={})
```

or maybe 

```python
df.style(rows={'A': [func1, func2], 'B': func2}, columns={})
```

What's the API for each `func`?

```python
def f(d, i, s)
    """
    d : any
        cell value
    i : int
        index location in row / column
    s : Series
        the row / column
    """
    pass
```

We could do something like `range`, is that too magical?

```python
def f(*args)
    pass
```

This should cover most cases. The only other thing is *maybe* the index.

In [3]:
df = pd.DataFrame({'A': np.linspace(1, 5, 5), 'B': np.random.randn(5)})
df

,A,B
0,1,-1.683868
1,2,-0.252138
2,3,-3.282090
3,4,1.783877
4,5,-1.314358


In [4]:
def is_positive_bg(s):
    colors = {True: ('backgroud-color', 'teal'),
              False: ('background-color', 'blue')}
    if isinstance(s, pd.Series):
        return (s > 0).map(colors)
    else:
        return (s > 0).applymap(lambda x: colors[x])

def is_even_font(s):
    colors = {True: ('font-color', 'red'),
              False: ('font-color', 'yellow')}
    if isinstance(s, pd.Series):
        return (s % 2).map(colors)
    else:
        return(s % 0).applymap(lambda x: colors[x])


In [5]:
is_positive_bg(df)

,A,B
0,"(backgroud-color, teal)","(background-color, blue)"
1,"(backgroud-color, teal)","(background-color, blue)"
2,"(backgroud-color, teal)","(background-color, blue)"
3,"(backgroud-color, teal)","(backgroud-color, teal)"
4,"(backgroud-color, teal)","(background-color, blue)"


In [6]:
df.style(columns={'A': is_even_font}, frame=[is_positive_bg])

AttributeError: 'DataFrame' object has no attribute 'style'

In [21]:
def monkeypatch_method(cls):
    def decorator(func):
        setattr(cls, func.__name__, func)
        return func
    return decorator

@monkeypatch_method(pd.DataFrame)
def style(self, rows=None, columns=None, frame=None):

    row_styles, col_styles, frame_styles = self._build_styles(
        rows=rows, columns=columns, frame=frame
    )

@monkeypatch_method(pd.DataFrame)
def _build_styles(self, rows=None, columns=None, frame=None):

    final = []  # (row, col, prop, value)
    
    if rows is not None:
        for row, func in rows.items():
            props, values  = zip(*func(self.loc[row]))
            final.append(pd.DataFrame({'row': row, 'col': self.columns,
                                       'prop': props, 'value': values}))
    if columns is not None:
        for col, func in columns.items():
            props, values  = zip(*func(self[col]))
            final.append(pd.DataFrame({'row': self.index, 'col': col,
                                       'prop': props, 'value': values}))

    if frame is not None:
        for func in frame:
            r = func(self)
            for col in r:
                try:
                    prop, values = zip(*r[col])
                except ValueError:
                    prop, values = '', ''
                    final.append(pd.DataFrame({'row': r.index, 'col': col,
                                           'prop': props, 'value': values}))

    r = pd.concat(final)[['row', 'col', 'prop', 'value']]
    r = (r['prop'] + ':' + r['value']).groupby([r['row'], r['col']]).agg(
        lambda x: '; '.join(x))
    
    return r

In [ ]:
pd.Series.apply

In [25]:
df.apply(maybe_is_positive_bg)

,A,B
0,,background-color: blue
1,,background-color: blue
2,,background-color: blue
3,,
4,,background-color: blue


In [29]:
df.head()

,A,B
0,1,-1.683868
1,2,-0.252138
2,3,-3.282090
3,4,1.783877
4,5,-1.314358


In [28]:
df.apply(maybe_is_positive_bg, axis=0)

,A,B
0,,background-color: blue
1,,background-color: blue
2,,background-color: blue
3,,
4,,background-color: blue


In [39]:
def f(x):
    if x >= 0:
        return 'background-color: blue'
    else:
        return ''

In [8]:
def maybe_is_positive_bg(s):
    colors = {True: '',
              False: 'background-color: blue'}
    if isinstance(s, pd.Series):
        return (s > 0).map(colors)
    else:
        return (s > 0).applymap(lambda x: colors[x])

In [9]:
{k: ";".join(v) for k, v in s.to_dict().items()}

NameError: name 's' is not defined

In [17]:
f = df._build_styles(columns={'A': is_even_font, 'B': is_positive_bg},
                     frame=[maybe_is_positive_bg])

In [19]:
f

row  col
0    A             font-color:red; background-color:
     B      background-color:blue; background-color:
1    A          font-color:yellow; background-color:
     B      background-color:blue; background-color:
2    A             font-color:red; background-color:
     B      background-color:blue; background-color:
3    A           font-color:yellow; backgroud-color:
     B        backgroud-color:teal; backgroud-color:
4    A             font-color:red; background-color:
     B      background-color:blue; background-color:
dtype: object

In [20]:
f.unstack()

col,A,B
row,,
0,font-color:red; background-color:,background-color:blue; background-color:
1,font-color:yellow; background-color:,background-color:blue; background-color:
2,font-color:red; background-color:,background-color:blue; background-color:
3,font-color:yellow; backgroud-color:,backgroud-color:teal; backgroud-color:
4,font-color:red; background-color:,background-color:blue; background-color:


In [147]:
from IPython import display

In [153]:
df.to_html()

'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th></th>\n      <th>A</th>\n      <th>B</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <th>0</th>\n      <td>1</td>\n      <td>0.215801</td>\n    </tr>\n    <tr>\n      <th>1</th>\n      <td>2</td>\n      <td>-1.262171</td>\n    </tr>\n    <tr>\n      <th>2</th>\n      <td>3</td>\n      <td>1.314150</td>\n    </tr>\n    <tr>\n      <th>3</th>\n      <td>4</td>\n      <td>-1.298183</td>\n    </tr>\n    <tr>\n      <th>4</th>\n      <td>5</td>\n      <td>0.514918</td>\n    </tr>\n  </tbody>\n</table>'

In [152]:
f.unstack()

col,A,B
row,,
0,font-color:red; backgroud-color:teal,backgroud-color:teal; backgroud-color:teal
1,font-color:yellow; background-color:teal,background-color:blue; background-color:blue
2,font-color:red; backgroud-color:teal,backgroud-color:teal; backgroud-color:teal
3,font-color:yellow; background-color:teal,background-color:blue; background-color:blue
4,font-color:red; backgroud-color:teal,backgroud-color:teal; backgroud-color:teal


In [139]:
df

,A,B
0,1,0.215801
1,2,-1.262171
2,3,1.314150
3,4,-1.298183
4,5,0.514918


In [89]:
f.unstack()

col,A,B
row,,
0,font-color:red; font-color:teal,backgroud-color:teal; font-color:teal
1,font-color:yellow; font-color:teal,background-color:blue; font-color:blue
2,font-color:red; font-color:teal,backgroud-color:teal; font-color:teal
3,font-color:yellow; font-color:teal,background-color:blue; font-color:blue
4,font-color:red; font-color:teal,background-color:blue; font-color:blue


In [125]:
r = pd.concat(f)[['row', 'col', 'prop', 'value']]

In [134]:
(r.prop + ':' + r.value).groupby([r.row, r.col]).agg(lambda x: '; '.join(x))

row  col
0    A              font-color:red; backgroud-color:teal
     B        backgroud-color:teal; backgroud-color:teal
1    A          font-color:yellow; background-color:teal
     B      background-color:blue; background-color:blue
2    A              font-color:red; backgroud-color:teal
     B        backgroud-color:teal; backgroud-color:teal
3    A          font-color:yellow; background-color:teal
     B      background-color:blue; background-color:blue
4    A              font-color:red; backgroud-color:teal
     B        backgroud-color:teal; backgroud-color:teal
dtype: object

In [128]:
r.groupby(['row', 'col'])

In [145]:
r

,row,col,prop,value
0,0,A,font-color,red
1,1,A,font-color,yellow
2,2,A,font-color,red
3,3,A,font-color,yellow
4,4,A,font-color,red
0,0,B,backgroud-color,teal
1,1,B,background-color,blue
2,2,B,backgroud-color,teal
3,3,B,background-color,blue
4,4,B,backgroud-color,teal


In [14]:
pd.DataFrame(np.where(df > 0, 'a', 'b'), index=df.index, columns=df.columns)

,A,B
0,a,a
1,a,b
2,a,a
3,a,b
4,a,b


In [18]:
df.B.replace

0    0.520844
1   -0.139787
2    0.723146
3   -0.025383
4   -0.999421
Name: B, dtype: float64

In [16]:
(df > 0).replace({True: 'a', False: 'b'})

,A,B
0,a,a
1,a,b
2,a,a
3,a,b
4,a,b
